# 08 - Concert Ranking Engine

## Goal

Build the first explainable GigRoute recommendation model by combining music preferences with geographic distance.

## Tasks

- Load concert and artist enrichment data
- Define test user preferences
- Filter concerts by date and travel radius
- Calculate artist preference score
- Calculate genre relevance score
- Calculate distance score
- Combine signals into a final ranking score
- Validate and explain recommendation ordering

In [5]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text

In [9]:
project_path = Path("..")

load_dotenv(project_path / ".env", override=True)

db_user = os.getenv("POSTGRES_USER")
db_password = os.getenv("POSTGRES_PASSWORD")
db_name = os.getenv("POSTGRES_DB")
db_port = os.getenv("POSTGRES_PORT")

## Database Connection

The PostgreSQL/PostGIS database is used to filter concerts geographically before recommendation scoring.

In [11]:
database_url = URL.create(
    drivername= "postgresql+psycopg2",
    username = db_user,
    password = db_password,
    host = "localhost",
    port = int(db_port),
    database=db_name
)

engine = create_engine(database_url)

In [17]:

# connection test

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM events")
    )
    event_count = result.scalar()

In [18]:
print("Events in database:", event_count)

Events in database: 1172


## Load Artist Enrichment Data

The final artist identity mapping and MusicBrainz genre data are loaded for recommendation scoring.

In [23]:
musicbrainz_path = (project_path / "data" / "processed" / "musicbrainz")

artist_mapping = pd.read_csv(
    musicbrainz_path / "artist_mapping.csv"
)

artist_genres = pd.read_csv(
    musicbrainz_path / "artist_genres.csv"
)

In [24]:
print("Artist mappings:", artist_mapping.shape)
print("Artist genres:", artist_genres.shape)

Artist mappings: (472, 19)
Artist genres: (785, 3)


In [25]:
artist_mapping["match_status"].value_counts()

match_status
auto_match         293
ambiguous           82
no_candidate        72
unresolved          13
secondary_match     12
Name: count, dtype: int64

## Test User Preferences

A reproducible test profile is used to validate the ranking logic before it is moved into the application.

In [26]:
user_latitude = 52.5200
user_longitude = 13.4050

radius_km = 300

start_date = "2026-08-21"
end_date = "2027-02-28"

In [27]:
preferred_artists = (
    artist_mapping.loc[
        artist_mapping["mbid"].notna(),
        "ticketmaster_artist_name"
    ]
    .drop_duplicates()
    .head(3)
    .tolist()
)

preferred_artists

['Heavysaurus', 'Rawayana', 'Dance Gavin Dance']

In [28]:
preferred_genres = (
    artist_genres["genre"]
    .value_counts()
    .head(3)
    .index
    .tolist()
)

preferred_genres

['pop', 'rock', 'hip hop']

## Geographic Candidate Selection

PostGIS filters events by travel radius and calculates the distance from the test user's location.

In [29]:
events_query = text("""
    SELECT
        event_id,
        event_name,
        artist_name,
        event_date,
        event_time,
        venue_name,
        city,
        country,
        latitude,
        longitude,
        ST_Distance(
            location,
            ST_SetSRID(
                ST_MakePoint(
                    :user_longitude,
                    :user_latitude
                ),
                4326
            )::geography
        ) / 1000 AS distance_km
    FROM events
    WHERE event_date BETWEEN :start_date AND :end_date
      AND ST_DWithin(
            location,
            ST_SetSRID(
                ST_MakePoint(
                    :user_longitude,
                    :user_latitude
                ),
                4326
            )::geography,
            :radius_meters
      )
    ORDER BY event_date;
""")

In [30]:
events_df = pd.read_sql(
    events_query,
    engine,
    params={
        "user_longitude": user_longitude,
        "user_latitude": user_latitude,
        "radius_meters": radius_km * 1000,
        "start_date": start_date,
        "end_date": end_date
    }
)

In [33]:
events_df.head()
events_df.shape

events_df["distance_km"].describe()

count    523.000000
mean     109.427239
std      118.528368
min        1.072681
25%        3.088968
50%        6.482444
75%      257.635170
max      295.985652
Name: distance_km, dtype: float64

In [34]:
print("Nearby events:", len(events_df))

print(
    "Maximum distance:",
    events_df["distance_km"].max()
)

Nearby events: 523
Maximum distance: 295.98565240878


In [35]:
print(
    "All events within radius:",
    events_df["distance_km"]
    .le(radius_km)
    .all()
)

All events within radius: True


## Enrich Events with Artist Identity

Nearby concerts are joined with the final artist mapping so that confidently resolved MusicBrainz identities can be used for genre-based recommendation scoring.

In [36]:
mapping_columns = artist_mapping[
    [
        "ticketmaster_artist_name",
        "mbid",
        "match_status"
    ]
].copy()

In [37]:
events_enriched = events_df.merge(
    mapping_columns,
    left_on="artist_name",
    right_on="ticketmaster_artist_name",
    how="left"
)

In [38]:
print("Events before merge:", len(events_df))
print("Events after merge:", len(events_enriched))

Events before merge: 523
Events after merge: 523


In [39]:
print(
    "Events with resolved artist identity:",
    events_enriched["mbid"].notna().sum()
)

print(
    "Events without resolved artist identity:",
    events_enriched["mbid"].isna().sum()
)

Events with resolved artist identity: 365
Events without resolved artist identity: 158


In [40]:
events_enriched["match_status"].value_counts(
    dropna=False
)

match_status
auto_match         358
ambiguous           87
no_candidate        58
unresolved          11
secondary_match      7
NaN                  2
Name: count, dtype: int64

In [41]:
genres_by_mbid = (
    artist_genres
    .groupby("mbid")["genre"]
    .apply(set)
    .to_dict()
)

In [42]:
list(genres_by_mbid.items())[:3]

[('015c8b7d-015b-4603-9792-18bb440178ac', {'hip hop', 'r&b'}),
 ('0211be88-dc0f-4002-a1de-4001bb359d51',
  {'alternative pop',
   'dark electro',
   'pop',
   'pop rock',
   'sexy drill',
   'synth-pop'}),
 ('033f413f-325f-43db-aeaa-89b0d77299cd', {'pop'})]

In [43]:
events_enriched["genres"] = (
    events_enriched["mbid"]
    .map(genres_by_mbid)
    .apply(
        lambda genres:
        genres if isinstance(genres, set)
        else set()
    )
)

In [44]:
events_enriched[
    [
        "artist_name",
        "mbid",
        "genres"
    ]
].head(20)

,artist_name,mbid,genres
0,Austra,af9e8d94-d471-4f91-82c7-250c22545162,"{indietronica, electronic, dark wave, indie po..."
1,Diljit Dosanjh,f931c961-b647-4861-be8c-f47d84a4de51,{bhangra}
2,Diljit Dosanjh,f931c961-b647-4861-be8c-f47d84a4de51,{bhangra}
3,Ticketmaster Suite Hamburg,NaN,{}
4,Zarna Garg,4b877353-2b15-432c-a439-f06cb210e033,{}
5,Pashanim,962a2590-91ed-4946-9724-f7dff6c61e03,"{hip hop, trap, pop rap}"
6,Sands Open Air Festival,NaN,{}
7,WE OUTSIDE,NaN,{}
8,Old but Gold Ü30 HipHop Party,NaN,{}
9,Hudson Freeman,34356aec-8c4b-4f15-997c-e972cdede64d,{}


In [45]:
preferred_artist_set = {
    artist.casefold()
    for artist in preferred_artists
}

In [46]:
events_enriched["artist_score"] = (
    events_enriched["artist_name"]
    .fillna("")
    .str.casefold()
    .isin(preferred_artist_set)
    .astype(float)
)

In [47]:
events_enriched["artist_score"].value_counts()

artist_score
0.0    520
1.0      3
Name: count, dtype: int64

In [48]:
preferred_genre_set = {
    genre.casefold()
    for genre in preferred_genres
}

In [49]:
def calculate_genre_score(
    event_genres,
    preferred_genres
):
    if not preferred_genres:
        return 0.0

    normalized_event_genres = {
        genre.casefold()
        for genre in event_genres
    }

    matching_genres = (
        normalized_event_genres
        & preferred_genres
    )

    return (
        len(matching_genres)
        / len(preferred_genres)
    )

In [50]:
events_enriched["genre_score"] = (
    events_enriched["genres"]
    .apply(
        lambda genres:
        calculate_genre_score(
            genres,
            preferred_genre_set
        )
    )
)

## Distance Score

Geographic distance is converted into a score between 0 and 1 so that closer concerts receive a higher recommendation score.

In [52]:
events_enriched["distance_score"] = (
    1
    - (
        events_enriched["distance_km"]
        / radius_km
    )
).clip(
    lower=0,
    upper=1
)

In [53]:
events_enriched["distance_score"].describe()

count    523.000000
mean       0.635243
std        0.395095
min        0.013381
25%        0.141216
50%        0.978392
75%        0.989703
max        0.996424
Name: distance_score, dtype: float64

In [54]:
events_enriched["distance_score"].between(0, 1).all()

np.True_

## Weighted Ranking Score

Artist preference, genre relevance, and travel distance are combined using configurable baseline weights.

In [55]:
artist_weight = 0.50
genre_weight = 0.30
distance_weight = 0.20

In [56]:
total_weight = (
    artist_weight
    + genre_weight
    + distance_weight
)

print("Total weight:", total_weight)

Total weight: 1.0


In [57]:
assert abs(total_weight - 1.0) < 1e-9

In [58]:
events_enriched["ranking_score"] = (
    events_enriched["artist_score"] * artist_weight
    + events_enriched["genre_score"] * genre_weight
    + events_enriched["distance_score"] * distance_weight
)

In [60]:
events_enriched["ranking_score"].describe()

count    523.000000
mean       0.160127
std        0.107389
min        0.002676
25%        0.049254
50%        0.197105
75%        0.198598
max        0.897304
Name: ranking_score, dtype: float64

In [62]:
events_enriched["ranking_score"].between(0, 1).all()

np.True_

In [63]:
ranked_events = (
    events_enriched
    .sort_values(
        ["ranking_score", "event_date"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

In [64]:
score_columns = [
    "distance_km",
    "genre_score",
    "distance_score",
    "ranking_score"
]

ranked_events[score_columns] = (
    ranked_events[score_columns]
    .round(3)
)

In [65]:
print(
    "Ranking sorted correctly:",
    ranked_events["ranking_score"]
    .is_monotonic_decreasing
)

Ranking sorted correctly: True


In [ ]:
print(
    "Ranking scores valid:",
    ranked_events["ranking_score"]
    .between(0, 1)
    .all()


Ranking scores valid: True


## Explain Top Recommendation

The highest-ranked concert is broken down into its individual scoring components to verify that the recommendation is understandable and explainable.

In [67]:
top_event = ranked_events.iloc[0]

print("Top recommendation")
print("------------------")
print("Event:", top_event["event_name"])
print("Artist:", top_event["artist_name"])
print("Date:", top_event["event_date"])
print("City:", top_event["city"])
print("Distance:", top_event["distance_km"], "km")

print()
print("Artist score:", top_event["artist_score"])
print("Genre score:", top_event["genre_score"])
print("Distance score:", top_event["distance_score"])
print("Final ranking score:", top_event["ranking_score"])

Top recommendation
------------------
Event: Dance Gavin Dance - Europe Tour 2026
Artist: Dance Gavin Dance
Date: 2026-09-03 00:00:00
City: Berlin
Distance: 4.043 km

Artist score: 1.0
Genre score: 0.667
Distance score: 0.987
Final ranking score: 0.897


In [68]:
print("Total ranked events:", len(ranked_events))

print(
    "Ranking sorted correctly:",
    ranked_events["ranking_score"].is_monotonic_decreasing
)

print(
    "Ranking scores valid:",
    ranked_events["ranking_score"]
    .between(0, 1)
    .all()
)

print(
    "Missing ranking scores:",
    ranked_events["ranking_score"].isna().sum()
)

print(
    "Events within radius:",
    ranked_events["distance_km"]
    .le(radius_km)
    .all()
)

Total ranked events: 523
Ranking sorted correctly: True
Ranking scores valid: True
Missing ranking scores: 0
Events within radius: True


## Findings

- Concert candidates were filtered by date range and travel radius using PostGIS.
- Artist identities and MusicBrainz genres were joined to nearby events without removing unresolved artists.
- Artist preference, genre relevance, and geographic distance were converted into comparable scores between 0 and 1.
- A configurable weighted ranking model combined the three recommendation signals.
- Ranking outputs were validated for score range, geographic constraints, missing values, and ordering.
- Individual score components remain visible, making each recommendation explainable.
- The ranking logic is ready to be refactored into reusable application code.